# Formula 1 racing predictions with Scikit-learn for model monitoring
This notebook builds and deploys a Snap Random Forest Regressor machine learning model using refined datasets based on the <a href="https://www.kaggle.com/datasets/jtrotman/formula-1-race-data" target="_blank">Formula 1 racing data</a>.

## Learning objectives
In this notebook, you will learn how to:

- Explore data
- Prepare data for training and evaluation
- Create a scikit-learn pipeline
- Train and evaluate a model
- Store a model in the Watson Machine Learning (WML) repository
- Deploy and score the model

## Contents
1. [Set up the project information](#setup)
2. [Read and explore data](#explore)
3. [Build a model](#build)
4. [Test the model](#test)
5. [Publish the model](#publish)
6. [Deploy and score](#deploy)

<a name="setup"></a>

## Set up the project information

To insert information about the project into the following cell, click the **Options** menu, and choose **Insert project token**.

<a name="explore"></a>

## Read and explore data

To read the training data from the project, follow these steps:
1. Click inside the following cell.
1. Click the **Code Snippets** icon.
2. Select **Read data**.
3. Click **Select data from project**.
4. Select **Data asset > f1_resultsTill24_points.csv**.
5. Click **Select**.
6. For *Load as*, select **pandas DataFrame**.
7. Click **Insert code to cell**.
8. Change the name of the dataframe from *df_1* to **data_df** to match the rest of the code in this notebook.

In [ ]:
print('Columns: ', list(data_df.columns))
print('Number of columns: ', len(data_df.columns))
print('Number of records: ', data_df.count())

<a name="build"></a>

## Build a model

### Prepare the data

The following cells perform these tasks:
- Define the features to use to train the model using the training data.
- Split the data into training and holdout data.

In [ ]:
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor

# Feature Selection
# Select the features that represent the grid position, the entities (driver/team/track), and performance metrics (momentum/history).
features = [
    'grid', 
    'driver_age', 
    'hist_track_points', 
    'momentum_3', 
    'year',
    'driverId', 
    'constructorId', 
    'circuitId'
]
target = 'points'

X = data_df[features]
y = data_df[target]

# Split the data into training and testing sets (80/20 split)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


### Train the model

The following cell initializes the Snap Random Forest Regressor and trains the model.

In [ ]:
# Initialize the Snap Random Forest Regressor
# n_jobs=-1 utilizes all available CPU cores for faster training
snap_rf = RandomForestRegressor(
    n_estimators=100,
    max_depth=10,
    n_jobs=-1,
    random_state=42
)

# Train the model
print("Training Snap Random Forest Regressor...")
snap_rf.fit(X_train.values, y_train.values)
print("Done")


### Evaluate the trained model

The following cell uses the holdout data to make predictions and determine the accuracy of those predictions.

In [ ]:
# Make predictions and evaluate
y_pred = snap_rf.predict(X_test.values)

mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"\nModel Evaluation:")
print(f"Mean Squared Error (MSE): {mse:.4f}")
print(f"R-squared Score (R2): {r2:.4f}")


### View the feature summary

The following cell shows the most important features used to train the model.

In [ ]:
# Optional: Show feature importance
import numpy as np
import matplotlib.pyplot as plt

importances = snap_rf.feature_importances_
indices = np.argsort(importances)[::-1]

print("\nFeature Importances:")
for f in range(X.shape[1]):
    print(f"{f + 1}. {features[indices[f]]} ({importances[indices[f]]:.4f})")

<a name="test"></a>

## Test the model

### Load some testing data
 
To read the testing data from the project, follow these steps:
1. Click inside the following cell.
1. Click the **Code Snippets** icon.
2. Select **Read data**.
3. Click **Select data from project**.
4. Select **Data asset > f1_resultsFrom25_testing.csv**.
5. Click **Select**.
6. For *Load as*, select **pandas DataFrame**.
7. Click **Insert code to cell**.
8. Change the name of the dataframe to **test_df** to match the rest of the code in this notebook.

### Select the features in the test data

The following cell identifies the same features in the test data that were used in the training data.

In [ ]:
# Select the exact same features used during training
# Note: Features must be in the same order as they were for snap_rf.fit()
test_features = [
    'grid', 
    'driver_age', 
    'hist_track_points', 
    'momentum_3', 
    'year',
    'driverId', 
    'constructorId', 
    'circuitId'
]

X_new = test_df[test_features]


### Generate the predictions

The following cell generates the predictions using the test data.

In [ ]:
# Use .values to ensure it's passed as a clean array to the model
predicted_points = snap_rf.predict(X_new.values)

# Add the predictions back to the dataframe for easy viewing
test_df['predicted_points'] = predicted_points

# Display the top predicted performers for a specific race
# Let's look at the first race in the test set (e.g., Australian Grand Prix)
first_race_name = test_df['circuit_name'].iloc[0]
race_results = test_df[test_df['circuit_name'] == first_race_name]

# Sort by predicted points to see the "predicted" podium
race_results = race_results.sort_values(by='predicted_points', ascending=False)

print(f"Predicted Results for: {first_race_name}")
print(race_results[['grid', 'driver_name', 'team_name', 'predicted_points']].head(10))

## Publish the model

### Set up the credentials

The following cells pass your API key for authentication.

In [ ]:
import getpass

api_key = getpass.getpass("Please enter your api key (press enter): ")

In [ ]:
from ibm_watsonx_ai import Credentials

credentials = Credentials(
    api_key=api_key,
    url='https://us-south.ml.cloud.ibm.com'
)

In [ ]:
from ibm_watsonx_ai import APIClient

client = APIClient(credentials)

### Set up the model metadata

The following cells specify the model name, deployment name, and software specifications for storing the model.

In [ ]:
MODEL_NAME = "Random Forest Regressor model to predict driver points"

DEPLOYMENT_NAME = "Random Forest Regressor model deployment to predict driver points"

In [ ]:
sk_version = "scikit-learn_1.6"

software_spec_id = client.software_specifications.get_id_by_name("runtime-25.1-py3.12")

In [ ]:

model_props = {
    client.repository.ModelMetaNames.NAME: "{}".format(MODEL_NAME),
    client.repository.ModelMetaNames.TYPE: sk_version,
    client.repository.ModelMetaNames.SOFTWARE_SPEC_ID: software_spec_id
}

### Set the deployment space

The following cells list the current deployment spaces so you can easily locate the space ID.

In [ ]:
client.spaces.list(limit=10)

In [ ]:
space_id = getpass.getpass("Please enter your space ID (press enter): ")

In [ ]:
client.set.default_space(space_id)

### Store the model

The following cell saves the model to the model repository.

In [ ]:
# Store the model
print("Storing model ...")
published_model_details = client.repository.store_model(
    model=snap_rf, 
    meta_props=model_props, 
    training_data=data_df[features], # Use only the features, not the whole dataframe
    training_target=data_df.points
)

model_id = client.repository.get_model_id(published_model_details)
print(f"Done! Model ID: {model_id}")

<a name="deploy"></a>

## Deploy and score

The following cell deploys the model to the specified deployment space.

In [ ]:
print("Deploying model...")
metadata = {
    client.deployments.ConfigurationMetaNames.NAME: DEPLOYMENT_NAME,
    client.deployments.ConfigurationMetaNames.ONLINE: {}
}
deployment = client.deployments.create(model_id, meta_props=metadata)
deployment_id = client.deployments.get_id(deployment)
    
print("Model id: {}".format(model_id))
print("Deployment id: {}".format(deployment_id))

In [ ]:
deployment_status = client.deployments.get_details(deployment_id)
print(deployment_status['entity']['status']['state'])

### Score the model

The following cells identifies the same features list in the testing data, formats the test data as a scoring payload, and then sends the scoring request.

In [ ]:
# Use the same 'features' list from your training script
# features = ['grid', 'driver_age', 'hist_track_points', 'momentum_3', 'year', 'driverId', 'constructorId', 'circuitId']
scoring_data = test_df[features]

In [ ]:
# Create the scoring payload
scoring_payload = {
    client.deployments.ScoringMetaNames.INPUT_DATA: [
        {
            "fields": scoring_data.columns.tolist(),
            "values": scoring_data.values.tolist()
        }
    ]
}

In [ ]:
print("Sending scoring request...")
predictions = client.deployments.score(deployment_id, scoring_payload)
print("Scoring complete.")

In [ ]:
print(test_df.columns)

In [ ]:
# Extract the predicted values
# The structure is usually predictions['predictions'][0]['values']
predicted_values = [val[0] for val in predictions['predictions'][0]['values']]

# Add the predictions to your dataframe for easy viewing
test_df['predicted_points'] = predicted_values

# Display the first few results
print(test_df[['grid', 'driver_name', 'team_name', 'predicted_points']].head(20))